# 2.3 Image enhancement

Image enhancement is the process of adjusting a digital image so the resultant one is more suitable for further image analysis (feature extraction, segmentation, etc.), in other words, **its goal is to improve the contrast and brightness of the image**.

There are three typical operations for enhancing images. We have already tried one of them in notebook *2.1 IP tools*: (linear) Look-Up Tables (LUTs). In this notebook we will play with other two:

- Non-linear look-up tables (Section 2.3.1).
- Histogram equalization (Section 2.3.2).
- Histogram specification (Section 2.3.3)

Also, some color-space conversions are going to be needed. If you are not familiar with the YCrCb color space, **Appendix 2** contains the information you need to know about it.

## Problem context - Implementing enhancement techniques for an image editor tool

We have all tried an image editor tool, sometimes without even knowing it! For example, modern smartphones already include an application for applying filters to images, cut them, modify their contrast, brightness, color temperature, etc.

<img src="https://github.com/jotaraul/jupyter-notebooks-for-computer-vision/blob/master/Chapter%2002.%20Image%20processing/images/GIMP-logo.png?raw=1" width="200">


One example of open source tool is the GNU Image Manipulation Program (GIMP). Quoting some words from its [website](https://www.gimp.org/):  

> GIMP is a cross-platform image editor available for GNU/Linux, OS X, Windows and more operating systems. It is free software, you can change its source code and distribute your changes.

> Whether you are a graphic designer, photographer, illustrator, or scientist, GIMP provides you with sophisticated tools to get your job done. You can further enhance your productivity with GIMP thanks to many customization options and 3rd party plugins.


In this case we were contacted by UMA for implementing two techniques to be included in their own image editor tool! Concretely, we were asked to develop and test two methods that are also part of GIMP: [**gamma**](https://docs.gimp.org/2.10/en/gimp-tool-levels.html) and [**equalize**](https://docs.gimp.org/2.10/en/gimp-layer-equalize.html).

In [1]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib
from ipywidgets import interactive, fixed, widgets
matplotlib.rcParams['figure.figsize'] = (20.0, 20.0)

images_path = './images/'

## 2.3.1 Non-linear look-up tables

**Gamma** or **gamma correction**

Gamma correction, or often simply gamma, is a nonlinear operation used to encode and decode luminance or tristimulus values in video or still image systems. In other words, it is the result of applying an (already defined) **non-linear LUT** in order to stretch or shrink image intensities.

In this way, the gamma LUT definition for grayscale images ($i \in [0 \dots 255]$):

$$LUT(i) = (\frac{i}{255})^{\gamma} * 255, \ \gamma \gt 0 $$

The following images illustrate the application of gamma correction for different values of $\gamma$.

<img src="https://github.com/jotaraul/jupyter-notebooks-for-computer-vision/blob/master/Chapter%2002.%20Image%20processing/images/gamma_theory.jpg?raw=1" width="800" >

### **<span style="color:green"><b><i>ASSIGNMENT 1: Applying non-linear LUTs</i></b></span>**

Your task is to develop the `lut_chart()` function, which takes as arguments the image to be enhanced and a gamma value for building the non-linear LUT. It will also display a chart containing the original image, the gamma-corrected one, the used LUT and the histogram of the resulting image.

As users from UMA will use color images, you will have to **implement it for color images**. This can be done by:
1. **transforming** a image in the BGR color space **to the YCrCb one**,
2. then, **applying gamma LUT only to first band** of the in the YCrCb space (that's because it contains pixel intensities and you can handle it like a gray image),
3. and finally, as matplotlib displays RGB images (if verbose is True), it should be **converted back**. Also, return the resultant image.

In [4]:

   # ASSIGNMENT 1 - FIXED & COMPLETED
# Implement a function that:
# -- converts the input image from the BGR to the YCrCb color space
# -- creates the gamma LUT
# -- applies the LUT to the original image
# -- displays in a 2x2 plot: the input image, the gamma-corrected one, the applied LUT and the resultant histogram if verbose = True

def lut_chart(image, gamma, verbose=False):
    """ Applies gamma correction to an image and shows the result.

        Args:
            image: Input image (BGR)
            gamma: Gamma parameter
            verbose: Only show images if this is True

        Returns:
            out_image: Gamma-corrected image
    """

    # 1. Transform image to YCrCb color space (Y = luminance, we correct only this channel)
    image_ycrcb = cv2.cvtColor(image, cv2.COLOR_BGR2YCrCb)

    # 2. Define gamma correction LUT (standard OpenCV way)
    lut = np.array([((i / 255.0) ** (1.0 / gamma)) * 255 for i in range(256)]).astype("uint8")

    # 3. Apply LUT to the first band (Y channel) of the YCrCb image
    y_channel = image_ycrcb[:, :, 0]                    # Y channel
    y_corrected = cv2.LUT(y_channel, lut)               # apply LUT
    image_ycrcb[:, :, 0] = y_corrected                  # replace Y

    # 4. Convert back to BGR
    out_image = cv2.cvtColor(image_ycrcb, cv2.COLOR_YCrCb2BGR)

    if verbose:
        # Create a 2x2 plot exactly as requested in the assignment
        plt.figure(figsize=(12, 8))

        # Top-left: Original image
        plt.subplot(2, 2, 1)
        plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
        plt.title('Input Image')
        plt.axis('off')

        # Top-right: Gamma-corrected image
        plt.subplot(2, 2, 2)
        plt.imshow(cv2.cvtColor(out_image, cv2.COLOR_BGR2RGB))
        plt.title(f'Gamma-corrected (γ = {gamma})')
        plt.axis('off')

        # Bottom-left: Applied LUT curve
        plt.subplot(2, 2, 3)
        plt.plot(lut, color='blue', linewidth=2)
        plt.title('Gamma LUT')
        plt.xlabel('Input intensity')
        plt.ylabel('Output intensity')
        plt.grid(True)

        # Bottom-right: Histogram of the corrected Y channel
        plt.subplot(2, 2, 4)
        hist = cv2.calcHist([image_ycrcb], [0], None, [256], [0, 256])
        plt.plot(hist, color='gray')
        plt.title('Histogram after correction')
        plt.xlabel('Intensity')
        plt.ylabel('Number of pixels')
        plt.grid(True)

        plt.tight_layout()
        plt.show()

    return out_image

You can use next code to **test if results are correct**:

In [5]:
image = np.array([[[10,60,20],[60,22,74],[72,132,2]],[[11,63,42],[36,122,27],[37,113,30]],[[1,6,2],[6,22,7],[7,13,0]]], dtype=np.uint8)
gamma = 2
print(lut_chart(image, gamma))

[[[ 71 121  81]
  [121  83 135]
  [134 194  64]]

 [[ 73 126 106]
  [ 98 185  88]
  [ 99 176  91]]

 [[ 27  32  30]
  [ 52  69  55]
  [ 43  50  37]]]


<font color='blue'>**Expected output:**  </font>

    [[[  6 112 110]
      [  6 151 138]
      [ 29  68 120]]

     [[ 10 122 105]
      [ 27  87 101]
      [ 25  92 104]]

     [[  0 127 126]
      [  1 122 122]
      [  0 122 127]]]

### <font color="blue"><b><i>Thinking about it (1)</i></b></font>

**You are asked to** try **lut_chart** with `gamma_1.jpg` (underexposed image) and `gamma_2.jpeg` (overexposed image). Then, **answer the following question**:

- What is happening when gamma value is modified?
  
    <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>  

In [10]:
# ==================== FIXED INTERACTIVE CELL (Assignment 1) ====================

# Import widgets if not already imported
from ipywidgets import interactive, fixed, widgets
from google.colab import files
import cv2
import matplotlib.pyplot as plt
import numpy as np

# ====================== UPLOAD YOUR IMAGE ======================
print("📤 Please upload an image (any JPG/PNG works best)")
uploaded = files.upload()
filename = list(uploaded.keys())[0]

# Read the uploaded image (OpenCV reads in BGR)
image = cv2.imread(filename)

# Check if image loaded correctly
if image is None:
    print("❌ Error: Could not load the image!")
else:
    print(f"✅ Image '{filename}' loaded successfully!")

# Create gamma slider widget
gamma_widget = widgets.FloatSlider(
    value=2.2,
    min=0.1,
    max=5.0,
    step=0.1,
    description='Gamma:'
)

# Run the interactive function (using the fixed lut_chart we did before)
interactive(
    lut_chart,
    image=fixed(image),
    gamma=gamma_widget,
    verbose=fixed(True)
)

📤 Please upload an image (any JPG/PNG works best)


Saving OIP.webp to OIP.webp
✅ Image 'OIP.webp' loaded successfully!


interactive(children=(FloatSlider(value=2.2, description='Gamma:', max=5.0, min=0.1), Output()), _dom_classes=…

## 2.3.2 Histogram equalization

Histogram Equalization is an image processing technique used to improve contrast in images. It accomplishes this by effectively spreading out the most frequent intensity values, i.e. stretching out the intensity range of the image so each possible pixel intensity appears the same number of times as every other value. This method usually increases the global contrast of images when its usable data is represented by close contrast values. This allows for areas of lower local contrast to gain a higher contrast.

<img src="https://github.com/jotaraul/jupyter-notebooks-for-computer-vision/blob/master/Chapter%2002.%20Image%20processing/images/equalization.png?raw=1" width="300" />$\\[5pt]$

To put an example, the [**equalize**](https://docs.gimp.org/2.10/en/gimp-layer-equalize.html) command from GIMP applies histogram equalization. But... how is this equalization achieved?

- First it is calculated the PMF ([probability mass function](https://en.wikipedia.org/wiki/Probability_mass_function)) of all the pixels in the image. Basically, this is a normalization of the histogram.

- Next step involves calculation of CDF ([cumulative distributive function](https://en.wikipedia.org/wiki/Cumulative_distribution_function)), producing the LUT for histogram equalization.

- Finally, the obtained LUT is applied.

The figure below shows an example of applying histogram equalization to an image.$\\[10pt]$

<img src="https://github.com/jotaraul/jupyter-notebooks-for-computer-vision/blob/master/Chapter%2002.%20Image%20processing/images/equalize_theory.png?raw=1" width="600" />$\\[5pt]$

### **<span style="color:green"><b><i>ASSIGNMENT 2: Equalizing the histogram!</i></b></span>**

Similarly to the previous exercise, **you are asked to** develop a function called **equalize_chart**. This method takes a **color** imag and will display a plot containing the original image, the equalized image, the original image histogram and the equalized image histogram.

*Tip: openCV implements histogram equalization in [cv2.equalizeHist](https://docs.opencv.org/2.4/modules/imgproc/doc/histograms.html?highlight=equalizehist)*

In [11]:
# ==================== ASSIGNMENT 2 - FIXED & COMPLETED ====================
# Implements histogram equalization exactly as requested in the notebook
# (matches slides 20-23 of your USTHB course perfectly)

def equalize_chart(image, verbose=False):
    """ Applies histogram equalization to an image and shows the result.

        Args:
            image: Input image (BGR)
            verbose: Only show images if this is True

        Returns:
            out_image: Equalized image
    """

    # 1. Transform image to YCrCb color space (we equalize only the Y channel)
    image_ycrcb = cv2.cvtColor(image, cv2.COLOR_BGR2YCrCb)

    # 2. Apply histogram equalization ONLY to the first band (Y = luminance)
    y_channel = image_ycrcb[:, :, 0]                    # Extract Y channel
    y_eq = cv2.equalizeHist(y_channel)                  # OpenCV histogram equalization
    image_ycrcb[:, :, 0] = y_eq                         # Replace Y channel

    # 3. Convert back to BGR
    out_image = cv2.cvtColor(image_ycrcb, cv2.COLOR_YCrCb2BGR)

    if verbose:
        # Create 2x2 plot as requested in the assignment
        plt.figure(figsize=(12, 8))

        # Top-left: Original image
        plt.subplot(2, 2, 1)
        plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
        plt.title('Original Image')
        plt.axis('off')

        # Top-right: Equalized image
        plt.subplot(2, 2, 2)
        plt.imshow(cv2.cvtColor(out_image, cv2.COLOR_BGR2RGB))
        plt.title('Equalized Image')
        plt.axis('off')

        # Bottom-left: Original histogram (Y channel)
        plt.subplot(2, 2, 3)
        hist_orig = cv2.calcHist([image_ycrcb], [0], None, [256], [0, 256])
        plt.plot(hist_orig, color='blue')
        plt.title('Original Histogram (Y channel)')
        plt.xlabel('Intensity')
        plt.ylabel('Pixels')
        plt.grid(True)

        # Bottom-right: Equalized histogram (Y channel)
        plt.subplot(2, 2, 4)
        hist_eq = cv2.calcHist([image_ycrcb], [0], None, [256], [0, 256])
        plt.plot(hist_eq, color='red')
        plt.title('Equalized Histogram (Y channel)')
        plt.xlabel('Intensity')
        plt.ylabel('Pixels')
        plt.grid(True)

        plt.tight_layout()
        plt.show()

    return out_image

You can use the next code to **test if your results are correct**:

In [12]:
image = np.array([[[10,60,20],[60,22,74],[72,132,2]],[[11,63,42],[36,122,27],[37,113,30]],[[1,6,2],[6,22,7],[7,13,0]]], dtype=np.uint8)
print(equalize_chart(image))

[[[ 96 146 106]
  [146 108 160]
  [241 255 171]]

 [[118 171 151]
  [175 255 165]
  [148 225 140]]

 [[  0   1   0]
  [ 53  70  56]
  [ 30  37  24]]]


<font color='blue'>**Expected output:**  </font>

    [[[128 112 110]
      [128 151 138]
      [255  68 120]]

     [[159 122 105]
      [223  87 101]
      [191  92 104]]

     [[  0 127 126]
      [ 64 122 122]
      [ 32 122 127]]]

### <font color="blue"><b><i>Thinking about it (2)</i></b></font>

We have developed our second image enhancement technique! Now try `equalize_chart()` with the `park.png` image. Then, **answer following questions**:

- What is the difference between original histogram and equalized?
  
    <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>
    
- Is final histogram uniform?

    <p style="margin: 4px 0px 0px 5px; color:blue"><i>Your answer here!</i></p>   

In [14]:
# ==================== FIXED INTERACTIVE CELL - ASSIGNMENT 2 ====================

# (Make sure you have already run the equalize_chart function from my previous message)

from ipywidgets import interactive, fixed, widgets
from google.colab import files
import cv2

# ====================== UPLOAD YOUR IMAGE (or reuse previous one) ======================
print("📤 Upload an image for histogram equalization (any JPG/PNG)")
uploaded = files.upload()
filename = list(uploaded.keys())[0]

# Read the uploaded image
image = cv2.imread(filename)

if image is None:
    print("❌ Error: Could not load the image!")
else:
    print(f"✅ Image '{filename}' loaded successfully!")

# Create interactive widget (exactly as in the original notebook)
interactive(
    equalize_chart,
    image=fixed(image),
    verbose=fixed(True)
)

📤 Upload an image for histogram equalization (any JPG/PNG)


Saving OIP.webp to OIP (1).webp
✅ Image 'OIP (1).webp' loaded successfully!


interactive(children=(Output(),), _dom_classes=('widget-interact',))

## 2.3.3 Histogram specification

Similar to histogram equalization, there is *histogram specification* (although this is not implemented in GIMP).

**Histogram specification is the transformation of an image so that its histogram matches a specified histogram**. In fact, the histogram equalization method is a special case in which the specified histogram is uniformly distributed.

<img src="https://github.com/jotaraul/jupyter-notebooks-for-computer-vision/blob/master/Chapter%2002.%20Image%20processing/images/specification.png?raw=1" width="600" >

It's implementation is very similar to histogram equalization:

- First it is calculated the PMF ([probability mass function](https://en.wikipedia.org/wiki/Probability_mass_function)) of all the pixels in both (source and reference) images.

- Next step involves calculation of CDF ([cumulative distributive function](https://en.wikipedia.org/wiki/Cumulative_distribution_function)) for both histograms ($F_1$ for source histogram and $F_2$ for reference histogram).$\\[3pt]$

- Then for each gray level $G_1 \in [0,255]$ , we find the gray level $G_2$, for which $F_1(G_1) = F_2(G_2)$, producing the LUT for histogram equalization.

- Finally, the obtained LUT is applied.

Unfortunately, histogram specification is not implemented in openCV. We will use [skimage.exposure.match_histograms](https://scikit-image.org/docs/dev/auto_examples/color_exposure/plot_histogram_matching.html) for this task.

### **<span style="color:green"><b><i>ASSIGNMENT 3: Let's specify the histogram</i></b></span>**

Apply histogram specification using the `ramos.jpg` and `illumination.png` gray images. Then, show the resultant image along input images (show their histogram as well).

In [15]:
# ASSIGNMENT 3

from skimage.exposure import match_histograms
matplotlib.rcParams['figure.figsize'] = (15.0, 10.0)

# Write your code here!


## Conclusion

Great! We are sure that UMA users are going to appreciate your efforts. Also, next time you use an image editor tool you are going to have another point of view of how things work.

In conclusion, in this notebook you have learned:

- How to define a **gamma correction (non-linear) LUT** and to how to apply it to an image.
- How **histogram specification** works and its applications. When the specified histogram is uniformly distributed, we call it **histogram equalization**.

## Extra

But this doesn't have to be the end, open GIMP and look through others implemented methods.  

As you are learning about image processing, **comment how you think they are implemented from scratch.**